# The findings ledger

The findings ledger contains statements about keys, dependencies, associations, and data quality. Each finding has a canonical statement and one or more evidence records. Detectors that support the same statement contribute evidence to that finding.

The `claim` is a readable summary. Programs can use `category`, `type`, `subject`, `specification`, and `evidence` to filter or inspect findings. Preserve the `id` and evidence when citing a finding.

## Anatomy of a finding

| Field | Type | What it holds |
|---|---|---|
| `id` | string | Content hash of the statement (`category-type-subject-specification`); stable across reruns and detector changes |
| `category` | string | The taxonomy entry: what kind of statement this is |
| `type` | string? | Sub-type where a category needs one (e.g. `functional`/`predictive` inside `dependencies`) |
| `subject` | dict | What the statement is about: `columns` (always), plus `inputs` and `target` for dependencies, `unit` for keys and duplicates |
| `specification` | dict | The logical form: `form` (e.g. `value_mapping`, `affine`), `conditions`/`outcome` predicates for rules, `definition` details (comparison, missing-value handling, role, calendar reference) |
| `claim` | string | One precise sentence, written to be quoted verbatim |
| `evidence` | list | One record per contributing detector: method, basis, scope, validation, metrics |
| `interestingness` | float | Deterministic score in $[0,1]$ that ranks the ledger |
| `interpretation` | string? | A suggested modeling or data-handling implication, when one exists |

The fixture extends the quickstart orders table with a concentrated channel and invoice numbers in two formats.

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(7)
n = 1500

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
order_id = list(range(n))
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]
external_ref[-2] = external_ref[1]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]
channel = ["web" if rng.random() < 0.99 else "app" for _ in range(n)]
invoice_no = [f"INV-{i:04d}" if i % 2 == 0 else f"{i:04d}" for i in range(n)]

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
    "channel": channel,
    "invoice_no": invoice_no,
})

report = onceover.profile(df)
print(report.overview["headline"])

## The statement taxonomy

The `category` describes the statement. Coverage records which detectors ran; evidence identifies the methods that support each finding.

```{admonition} The taxonomy is evolving
:class: warning

The statement taxonomy — categories, types, and specification forms — is
evolving and is likely to change in future releases. Programs that consume
reports should read `schema_version`, match categories and types
defensively, and treat unknown values as opaque rather than fatal. Schema
0.2 itself replaced the flat finding-kind strings of 0.1, breaking
consumers that matched on them.
```

| Category | Type(s) | What the statement says | Detector(s) |
|---|---|---|---|
| `missingness` | — | a column is entirely null | `column_profile` |
| `concentration` | — | a column is constant, nearly constant, or heavily dominated by few values | `column_profile` |
| `column_roles` | — | a numeric column behaves like a category; a column looks like an identifier (inferred, not verified) | `column_profile` |
| `uniqueness` | — | a column or column set identifies rows: primary key, composite key, or near-unique with repeats | `keys` |
| `duplicates` | — | rows repeat earlier rows exactly | `row_duplicates` |
| `formats` | — | a string column mixes value formats | `column_profile` |
| `dependencies` | `functional` | one column is a (near-)function of others: a value mapping, an affine formula, or a piecewise-linear regime | `functional_dependencies`, `identities`, `regimes` |
| `dependencies` | `predictive` | a target is predictable from inputs, with named validation | `gbm`, `decision_tables`, `regimes` |
| `conditional_rules` | `if_then` | when predicates hold, an outcome holds, with measured rate and lift | `prism`, `association_rules` |
| `associations` | `statistical` | numeric columns move together (Pearson, Spearman, ξ) | `numeric_dependence` |
| `gaps` | — | a temporal column is missing expected values, against a disclosed calendar reference | `dates` |
| `intervals` | — | a temporal column has irregular spacing | `dates` |
| `ordering` | — | a column is sorted with row order | `sortedness` |
| `changes` | `residual_level` | a target's residual level shifts after a date | `regimes` |

The current distribution, as a histogram over this table's ledger:

In [ ]:
from collections import Counter

print(Counter((f["category"], f.get("type")) for f in report.findings))

## The evidence list

A finding can have evidence from several detectors. Each evidence record has five parts:

| Part | Fields | What they hold |
|---|---|---|
| `method` | `detector`, `algorithm`, `provenance` | who found it, with which algorithm and implementation source |
| `basis` | `measured` \| `estimated` \| `inferred` | counted or verified directly; computed from a sample or a model; or a rule-of-thumb reading of measurements |
| `scope` | `sampling` (`full`\|`sample`), `population_rows`, `rows_evaluated`, `missing_values` | over how much of the table, and how nulls were treated |
| `validation` | `method`, `training_rows`, `evaluation_rows`, `folds`, `used_for_selection` | whether the numbers come from direct comparison, a holdout, cross-validation, or a diagnostic partition |
| `metrics` | free-form | the numbers backing the claim: violations, rates, coefficients, effect sizes |

`basis` describes the support: a direct measurement, an estimate, or an inference. `validation` describes how the claim was checked. For example, a functional dependency can have measured counts and a direct comparison; a predictive dependency can have an estimated score from a holdout.

A conditional rule may have one record from its discovery detector and another from the shared evaluator. Each record retains its own scope and metrics.

## One statement, several detectors

On this table, the predictability screen and decision-table search
produce the same predictive statement about `region_id`. The ledger
keeps one statement with both evidence records:

In [ ]:
merged = next(f for f in report.findings
              if f["category"] == "dependencies"
              and len(f["evidence"]) > 1)
print(merged["claim"])
for ev in merged["evidence"]:
    m = ev["method"]
    print(f"  {m['detector']:16s} basis={ev['basis']:9s} "
          f"validation={ev['validation']['method']}")

The two detectors measured different things — a holdout accuracy and a
cross-validated lookup accuracy — and both records survive untouched; the
ledger never averages or pools evidence. Statements merge when their
category, type, subject, and specification match. The displayed claim
comes from the record with broader evaluated coverage.

Conditional rules also carry separate evidence records. PRISM
discovers a rule on data it may have used to grow it; a shared evaluator
then re-counts every rule on a deterministic train/evaluation partition
that selection reuses. One rule, two evidence records:

In [ ]:
rule = next(f for f in report.findings if f["category"] == "conditional_rules")
print(rule["claim"])
for ev in rule["evidence"]:
    print(f"  {ev['method']['detector']:16s} basis={ev['basis']:9s} "
          f"validation={ev['validation']['method']}")
print("  evaluated metrics:", {k: v for k, v in rule["evidence"][-1]["metrics"].items()
                                if k in ("conditional_rate", "baseline_rate", "lift", "joint_count")})

The claim uses the shared evaluator's counts. The PRISM discovery record remains attached as separate evidence.

## Ids and determinism

Finding IDs hash the category, type, subject, and specification. They do not depend on rank or detector, and they stay stable for the same statement. IDs are not sequential; treat them as opaque strings.

## Ranking: interestingness

The `interestingness` score uses four components:

$$\text{interestingness} = 0.45 \cdot \text{quality} + 0.35 \cdot
\text{strength} + 0.10 \cdot \text{specificity} + 0.10 \cdot \text{simplicity}$$

- **quality** — how the evidence was established: 1.0 for direct
  comparison on full data; 0.8 for holdouts, cross-validation, and
  diagnostic partitions; below that, by basis (measured 0.85, estimated
  0.6, inferred 0.4).
- **strength** — per category: for functional dependencies, the share of
  rows that obey the mapping ($1 - \text{violations}/n$); for predictive
  dependencies, held-out skill above baseline; for rules, a blend of
  rate-gain and lift capped by support; for associations, the largest
  measured effect.
- **specificity** — does the statement have a concrete logical form
  (1.0), or is it a bare observation (0.5)?
- **simplicity** — fewer inputs and conditions score higher:
  $1/(1 + (\text{inputs} + \text{conditions})/8)$.

For the quickstart's exact functional dependency, the score is 0.9889:
quality 1.0 (direct comparison, full data), strength 1.0 (zero violations
in 2,000 rows), specificity 1.0 (a `value_mapping` form), simplicity
$1/(1 + 1/8) = 0.889$. The [evidence chapter](../methodology/evidence.ipynb) derives the score from these fields.

## Selection: from candidates to a report

A table may produce hundreds of candidates. The report ranks them and applies these limits:

| Policy knob | Default | Effect |
|---|---:|---|
| `max_findings` | 100 | report-wide cap after ranking |
| `max_rules` | 20 | cap on conditional rules |
| `rules_per_outcome` | 2 | at most this many rules per outcome column |
| `predictive_findings_per_target` | 3 | at most this many predictive statements per target |

The Python API exposes `max_findings` and `max_rules`; the Rust
`SelectionPolicy` exposes the full set. The counts travel with the report —
`report.data["selection"]` records how many candidates, verified
candidates, distinct statements, and selected findings the run produced —
and the `rule_evaluation` coverage entry reports how many rule candidates
passed the common diagnostic partition.

## Groups

`report.groups` links findings that share columns — connected components
over the subjects of dependencies, rules, and associations. On the
quickstart table, `quantity` and `double_quantity` appear in one group
with the statements about their relationship. Assess redundancy from
the direction, form, and evidence of each finding.

## Filtering the ledger

The ledger is a plain list of dicts, so anything expressible in Python
works. Four patterns cover most needs.

By category — "give me the structure claims":

In [ ]:
structure = [f for f in report.findings
             if f["category"] in ("uniqueness", "duplicates", "dependencies",
                                  "conditional_rules")]
print(len(structure), "structural findings")
for f in structure[:5]:
    print(f"  [{f['category']}/{f.get('type', '-')}] {f['claim'][:70]}")

By subject column — "what does the report know about `quantity`?". The
subject's `columns` list is the canonical place to look:

In [ ]:
for f in report.findings:
    if "quantity" in f["subject"]["columns"]:
        print(f"[{f['category']}/{f.get('type', '-')}] {f['claim'][:80]}")

By basis — "which statements were directly measured on full data?"

In [ ]:
measured = [f for f in report.findings
              if any(ev["basis"] == "measured" and ev["scope"]["sampling"] == "full"
                     for ev in f["evidence"])]
print(len(measured), "statements measured on full data")

By validation method — look for `diagnostic_holdout`, `cross_validation`,
or `diagnostic_partition` in the evidence. These methods provide
different checks; a diagnostic partition may include discovery rows.

## What to act on

The `interpretation` field suggests possible actions, such as checking
duplicates before joining, inspecting a mapping before treating a column
as derived, or treating a numeric column as categorical. Use the
evidence and application context to decide whether an action is appropriate.

| Evidence pattern | How to use it |
|---|---|
| `measured` + `direct_comparison` + `full` sampling | Check candidate constraints, join keys, or derived-column rules against the current table and intended semantics |
| `measured` + `diagnostic_partition` (rules) | Read rate, lift, and support on the stated partition; discovery may have used those rows |
| `estimated` + `diagnostic_holdout` / `cross_validation` | Review predictive performance under the reported split or folds; check for leakage and transfer to new data |
| `inferred` | An interpretation to check, such as a suggested column role |

```{admonition} Check coverage when a finding is absent
:class: warning

A detector may miss a property because it was capped, sampled, skipped,
or not selected. Coverage records the work performed and its limits;
it does not establish why a particular property was missed.
[](coverage.ipynb) explains the statuses.
```

## Choosing detectors

Every part of the ledger so far comes from built-in detectors, and they
are selectable. `onceover.available_detectors()` lists them with the
categories they produce and the detectors they require; `profile(df,
detectors=[...])` selects a subset:

In [ ]:
for d in onceover.available_detectors()[:4]:
    print(f"{d['id']:24s} categories={d['categories']} requires={d['requires']}")

subset = onceover.profile(df, detectors=["keys", "functional_dependencies"])
print(subset.overview["headline"])

A selected detector may require another detector to build a shared resource. For example, decision tables use target profiles from `gbm`. The required detector runs, but only the selected detector contributes findings. Coverage labels the resource pass as required and other deselected passes as not enabled.